# Testing New Ephemeris and SPICE loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import spiceypy as spice

In [73]:
from contigo.solar_system_ephem import SPICEEphem

In [74]:
SPICEEphem()

INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\de440s.bsp
INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\naif0012.tls
INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\earth_latest_high_prec.bpc


In [7]:
sw_e = pd.read_hdf("./data/ESA_pod.hdf")

In [ ]:
body = ['SUN','MOON']

j2000 = pd.Timestamp('2000-01-01 12:00:00')
spj2000 = ((sw_e['DateTime'] - j2000).dt.total_seconds()).to_list()

# set all needed attributes
et = np.array([spice.unitim(sp_in,'GPS','ET') for sp_in in spj2000])

unique_et, inv = np.unique(et, return_inverse=True)

# get the body positions in ecef
bd_ecef = np.array([spice.spkpos(bd,unique_et,'ITRF93','NONE','EARTH')[0]
            for bd in body])

bd_full = bd_ecef[:,inv,:]

In [75]:
x = SPICEEphem()

INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\de440s.bsp
INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\naif0012.tls
INFO:Kernel already loaded - D:\GitHub\contigo_edr\contigo\data\earth_latest_high_prec.bpc


In [42]:
c_et, c_bd = x(body=['SUN','MOON'], et=et)

In [68]:
np.array_equal(c_bd,bd_full)

True

In [100]:
from contigo.solar_system_ephem import SolarSystemEnvironment

In [101]:
env = SolarSystemEnvironment(['SUN','MOON'], et=et[0:500], tolerance=1.,provider=x)

In [103]:
r_et, r_pos = env.get_ephem(et[0:500])
print(r_et.shape)
print(r_pos.shape)

(500,)
(2, 500, 3)


In [104]:
r_et, r_pos = env.get_ephem(et[0:1500])
print(r_et.shape)
print(r_pos.shape)

(1500,)
(2, 1500, 3)


In [105]:
r_et, r_pos = env.get_ephem(et)
print(r_et.shape)
print(r_pos.shape)

(276480,)
(2, 276480, 3)


In [ ]:
def _quantize(tolerance, t):
        """Quantize time using tolerance and return integer bin."""
        if tolerance == 0.0:
            # exact integer seconds binning
            return int(round(t))
        return int(round(t / tolerance))

def _quantize_numpy(tolerance, t):
        """Quantize time using tolerance and return integer bin."""
        if tolerance == 0.0:
            # exact integer seconds binning
            return int(np.round(t))
        return int(np.round(t / tolerance))